In [ ]:
import pandas as pd
from pathlib import Path

# -----------------------
# Google Drive path
# -----------------------
DRIVE = Path(r"G:\Meu Drive")

EXCEL_FILE = (
    DRIVE
    / "Anotacoes"
    / "Budget e Impostos"
    / "2025 ldn"
    / "Gastos totais.xlsx"
)

# -----------------------
# Load Excel file
# -----------------------
excel_sheets = pd.read_excel(
    EXCEL_FILE,
    sheet_name=["Mov_sem_internas", "Payslip"]
)

df = excel_sheets["Mov_sem_internas"].copy()
payslip_df = excel_sheets["Payslip"].copy()

# Save both sheets as CSV files
output_dir = Path.cwd() / "csv_exports"
output_dir.mkdir(exist_ok=True)

mov_csv = output_dir / "gastos_totais_mov_sem_internas.csv"
payslip_csv = output_dir / "gastos_totais_payslip.csv"

df.to_csv(mov_csv, index=False, encoding="utf-8-sig")
payslip_df.to_csv(payslip_csv, index=False, encoding="utf-8-sig")

print("✅ CSVs salvos em:")
print("  -", mov_csv)
print("  -", payslip_csv)

# -----------------------
# Clean data
# -----------------------
df["Date"] = pd.to_datetime(df["Date"])
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# Keep only expenses
df = df[df["Direction"] == "DEBIT"].copy()

# Positive expense values
df["Gasto"] = df["Amount"].abs()

print("✅ Data loaded successfully")
print("📂 File:", EXCEL_FILE)
print("📊 Shape:", df.shape)

print(df.head())

✅ Data loaded successfully
📂 File: G:\Meu Drive\Anotacoes\Budget e Impostos\2025 ldn\Gastos totais.xlsx
📊 Shape: (1177, 11)
        Date    Month                                        Description  \
0 2025-05-05  2025-05  Card transaction of 40.32 GBP issued by Boots,...   
1 2025-05-05  2025-05  Card transaction of 36.36 GBP issued by Sainsb...   
2 2025-05-06  2025-05  Card transaction of 16.39 GBP issued by Pret A...   
3 2025-05-06  2025-05  Card transaction of 7.80 GBP issued by Tesco S...   
4 2025-05-07  2025-05  Card transaction of 14.16 GBP issued by Amazon...   

   Amount Direction    Categoria TxnType Source  mes    year  Gasto  
0  -40.32     DEBIT     Farmácia    CARD   Wise    5  2025.0  40.32  
1  -36.36     DEBIT      Mercado    CARD   Wise    5  2025.0  36.36  
2  -16.39     DEBIT  Restaurante    CARD   Wise    5  2025.0  16.39  
3   -7.80     DEBIT      Mercado    CARD   Wise    5  2025.0   7.80  
4  -14.16     DEBIT       Amazon    CARD   Wise    5  2025.0  14.16  

In [2]:
# Instalar pacotes (se ainda não tiver rodado antes)
# !pip install pandas plotly openpyxl

import pandas as pd
import plotly.express as px
from IPython.display import display

# Categorias consideradas custos fixos
CATEGORIAS_FIXAS = ["Aluguel", "Home Box", "Transporte", "Internet", "Celular", "Casa", "Academia", "Mercado", "Restaurante"]


# ----------------------
# 1) Carregar dados
# ----------------------
def carregar_gastos(caminho_arquivo, aba="Mov_sem_internas"):
    df = pd.read_excel(caminho_arquivo, sheet_name=aba)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df = df[df["Direction"] == "DEBIT"].copy()   # só gastos
    df["Gasto"] = -df["Amount"]                  # gasto positivo
    return df


# ----------------------
# 2) Filtrar por período
# ----------------------
def filtrar_periodo(df, data_ini, data_fim):
    data_ini = pd.to_datetime(data_ini)
    data_fim = pd.to_datetime(data_fim)
    return df[(df["Date"] >= data_ini) & (df["Date"] <= data_fim)].copy()


# ----------------------
# 3) Gastos por período (D, W, M, Q, Y) - gráfico + tabela com valores inteiros
# ----------------------
def gastos_por_periodo(df, data_ini, data_fim, freq="M"):
    """
    freq: 'D' (dia), 'W' (semana), 'M' (mês), 'Q' (trimestre), 'Y' (ano)
    """
    sub = filtrar_periodo(df, data_ini, data_fim)
    sub = sub.set_index("Date")
    resumo = sub["Gasto"].resample(freq).sum().reset_index()
    resumo.rename(columns={"Date": "Periodo"}, inplace=True)

    # Arredonda Gasto para 0 casas decimais
    resumo["Gasto"] = resumo["Gasto"].round(0).astype(int)

    fig = px.line(
        resumo,
        x="Periodo",
        y="Gasto",
        markers=True,
        text="Gasto",
        title=f"Gastos ({freq}) de {data_ini} a {data_fim}",
        labels={"Periodo": "Período", "Gasto": "Gasto (GBP)"}
    )
    fig.update_traces(texttemplate="£%{text:.0f}", textposition="top center")
    fig.update_layout(hovermode="x unified")
    fig.show()

    return resumo


# ----------------------
# 4) Gastos por categoria (com TOTAL e %; tudo arredondado)
# ----------------------
def gastos_por_categoria(df, data_ini, data_fim, incluir_total=True):
    sub = filtrar_periodo(df, data_ini, data_fim)

    # soma por categoria
    resumo = (
        sub.groupby("Categoria", as_index=False)["Gasto"]
        .sum()
        .sort_values("Gasto", ascending=False)
    )

    # total (valor bruto, antes de arredondar)
    total_periodo = resumo["Gasto"].sum()

    # porcentagem em relação ao total
    resumo["Percent (%)"] = (resumo["Gasto"] / total_periodo * 100)

    # arredonda gastos e % para 0 casas decimais
    resumo["Gasto"] = resumo["Gasto"].round(0).astype(int)
    resumo["Percent (%)"] = resumo["Percent (%)"].round(0).astype(int)

    if incluir_total:
        linha_total = pd.DataFrame({
            "Categoria": ["TOTAL"],
            "Gasto": [int(round(total_periodo, 0))],
            "Percent (%)": [100]
        })
        resumo = pd.concat([resumo, linha_total], ignore_index=True)

    # gráfico (sem a linha TOTAL)
    fig = px.bar(
        resumo[resumo["Categoria"] != "TOTAL"],
        x="Categoria",
        y="Gasto",
        title=f"Gastos por categoria de {data_ini} a {data_fim}",
        text="Gasto",
        labels={"Categoria": "Categoria", "Gasto": "Gasto (GBP)"}
    )
    fig.update_traces(texttemplate="£%{text:.0f}", textposition="outside")
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

    return resumo, total_periodo


# ----------------------
# 5) Custos fixos x variáveis (com TOTAL e %; tudo arredondado)
# ----------------------
def custos_fixos_e_variaveis(df, data_ini, data_fim, incluir_total=True):
    """
    Retorna:
      - tabela_fixos, total_fixos
      - tabela_variaveis, total_variaveis
    com colunas:
      - Categoria
      - Gasto (arredondado sem decimais)
      - Percent (%) (arredondado sem decimais)
    """
    sub = filtrar_periodo(df, data_ini, data_fim).copy()

    # marca tipo de custo
    sub["Tipo_custo"] = sub["Categoria"].apply(
        lambda c: "Fixo" if c in CATEGORIAS_FIXAS else "Variável"
    )

    # ---------- FIXOS ----------
    df_fixos = sub[sub["Tipo_custo"] == "Fixo"].copy()
    if not df_fixos.empty:
        tabela_fixos = (
            df_fixos.groupby("Categoria", as_index=False)["Gasto"]
            .sum()
            .sort_values("Gasto", ascending=False)
        )
        total_fixos = tabela_fixos["Gasto"].sum()

        # porcentagem dentro do total de fixos
        tabela_fixos["Percent (%)"] = (tabela_fixos["Gasto"] / total_fixos * 100)

        # arredonda
        tabela_fixos["Gasto"] = tabela_fixos["Gasto"].round(0).astype(int)
        tabela_fixos["Percent (%)"] = tabela_fixos["Percent (%)"].round(0).astype(int)

        if incluir_total:
            linha_total_f = pd.DataFrame({
                "Categoria": ["TOTAL FIXOS"],
                "Gasto": [int(round(total_fixos, 0))],
                "Percent (%)": [100]
            })
            tabela_fixos = pd.concat([tabela_fixos, linha_total_f], ignore_index=True)
    else:
        tabela_fixos = pd.DataFrame(columns=["Categoria", "Gasto", "Percent (%)"])
        total_fixos = 0.0

    # ---------- VARIÁVEIS ----------
    df_var = sub[sub["Tipo_custo"] == "Variável"].copy()
    if not df_var.empty:
        tabela_variaveis = (
            df_var.groupby("Categoria", as_index=False)["Gasto"]
            .sum()
            .sort_values("Gasto", ascending=False)
        )
        total_variaveis = tabela_variaveis["Gasto"].sum()

        # porcentagem dentro do total de variáveis
        tabela_variaveis["Percent (%)"] = (tabela_variaveis["Gasto"] / total_variaveis * 100)

        # arredonda
        tabela_variaveis["Gasto"] = tabela_variaveis["Gasto"].round(0).astype(int)
        tabela_variaveis["Percent (%)"] = tabela_variaveis["Percent (%)"].round(0).astype(int)

        if incluir_total:
            linha_total_v = pd.DataFrame({
                "Categoria": ["TOTAL VARIÁVEIS"],
                "Gasto": [int(round(total_variaveis, 0))],
                "Percent (%)": [100]
            })
            tabela_variaveis = pd.concat([tabela_variaveis, linha_total_v], ignore_index=True)
    else:
        tabela_variaveis = pd.DataFrame(columns=["Categoria", "Gasto", "Percent (%)"])
        total_variaveis = 0.0

    return tabela_fixos, total_fixos, tabela_variaveis, total_variaveis


# ----------------------
# 6) Gasto diário por categoria - gráfico + registros detalhados
# ----------------------
def gastos_diarios_categoria(df, categoria, data_ini, data_fim):
    """
    categoria: string ("Restaurante") ou lista de strings (["Restaurante","Mercado"])

    Mostra:
      - gráfico do gasto diário (valores inteiros)
      - tabela com registros detalhados:
            Date, Description, Categoria, Amount, TxnType, Source

    Retorna:
      - DataFrame com os registros detalhados filtrados
    """
    sub = filtrar_periodo(df, data_ini, data_fim).copy()

    # garante lista de categorias
    if isinstance(categoria, str):
        categorias = [categoria]
    else:
        categorias = list(categoria)

    sub = sub[sub["Categoria"].isin(categorias)].copy()

    if sub.empty:
        print(f"Nenhum gasto encontrado em {categorias} entre {data_ini} e {data_fim}.")
        return pd.DataFrame(columns=["Date", "Description", "Categoria", "Amount"])

    # resumo diário para o gráfico
    resumo_diario = (
        sub.groupby("Date", as_index=False)["Gasto"]
        .sum()
        .sort_values("Date")
    )

    # arredonda Gasto diário
    resumo_diario["Gasto"] = resumo_diario["Gasto"].round(0).astype(int)

    total_periodo = resumo_diario["Gasto"].sum()
    label_cat = ", ".join(categorias)

    print(f"📅 Gasto diário em [{label_cat}] de {data_ini} a {data_fim}")
    print(f"💰 Total no período (aprox): £{total_periodo:.0f}\n")

    # gráfico interativo
    fig = px.bar(
        resumo_diario,
        x="Date",
        y="Gasto",
        title=f"Gasto diário em {label_cat} ({data_ini} a {data_fim})",
        labels={"Date": "Data", "Gasto": "Gasto (GBP)"},
        text="Gasto"
    )
    fig.update_traces(texttemplate="£%{text:.0f}", textposition="outside")
    fig.update_layout(hovermode="x unified")
    fig.show()

    # registros detalhados para inspeção
    registros = (
        sub[["Date", "Description", "Categoria", "Amount"]]
        .sort_values("Date")
        .reset_index(drop=True)
    )

    print("📋 Registros detalhados do período:\n")
    display(registros)

    return registros


In [3]:
data_ini = "2026-04-01"
data_fim = "2026-04-30"

# Por categoria
tabela_cat, total_cat = gastos_por_categoria(df, data_ini, data_fim)
display(tabela_cat)

# Fixos x variáveis
t_fixos, total_fixos, t_var, total_var = custos_fixos_e_variaveis(df, data_ini, data_fim)
display(t_fixos)
display(t_var)

# Diário em Restaurante (com registros detalhados)
reg_rest = gastos_diarios_categoria(df, "Viagem", data_ini, data_fim)


,Categoria,Gasto,Percent (%)
0,Aluguel,1598,35
1,Viagem,873,19
2,Restaurante,654,14
3,Amazon,412,9
4,Ingresso,182,4
5,Mercado,176,4
6,Transporte,125,3
7,Academia,109,2
8,Casa,104,2
9,Limpeza,80,2


,Categoria,Gasto,Percent (%)
0,Aluguel,1598,57
1,Restaurante,654,23
2,Mercado,176,6
3,Transporte,125,4
4,Academia,109,4
5,Casa,104,4
6,Celular,35,1
7,Home Box,2,0
8,TOTAL FIXOS,2804,100


,Categoria,Gasto,Percent (%)
0,Viagem,873,51
1,Amazon,412,24
2,Ingresso,182,11
3,Limpeza,80,5
4,Uber,64,4
5,Consulta,42,2
6,TV,38,2
7,Cabelo,25,1
8,TOTAL VARIÁVEIS,1715,100


📅 Gasto diário em [Viagem] de 2026-04-01 a 2026-04-30
💰 Total no período (aprox): £873



📋 Registros detalhados do período:



,Date,Description,Categoria,Amount
0,2026-04-14,British Airways,Viagem,-40.00
1,2026-04-14,British Airways,Viagem,-40.00
2,2026-04-14,British Airways,Viagem,-698.29
3,2026-04-17,Drivalia Recharges,Viagem,-95.00
